# Tokenization and vocabulary building

This notebook walks through a simple text-processing pipeline: downloading a corpus, inspecting the text, splitting it into tokens, building a vocabulary, and implementing a lightweight tokenizer.

In [2]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x1047cbed0>)

## 1) Download the sample corpus

This first step fetches the text file used in the chapter so the later examples can operate on real data.

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## 2) Read and inspect the raw text

We load the downloaded file and print its size and a short preview to confirm the data looks as expected.

In [4]:
import re
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


## 3) Basic whitespace tokenization

This example splits text by spaces only, which is the simplest possible tokenization strategy.

In [5]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


## 4) Separate punctuation from words

Here we include punctuation as standalone tokens so we can see how token boundaries change when special characters are preserved.

In [6]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


## 5) Clean the token list

This step removes empty strings and keeps only meaningful text pieces before proceeding to vocabulary construction.

In [10]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


## 6) Prepare the full corpus for tokenization

Now that the small example works, we apply the same regex-based preprocessing to the full text and prepare the vocabulary.

In [11]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


## 7) Preprocess the full text

This cell tokenizes the entire corpus and prints the total number of processed tokens.

In [8]:
print(preprocessed[:30])


['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 8) Inspect the first tokens

This cell shows the beginning of the preprocessed token list so we can visually verify the token boundaries.

In [9]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


## 9) Build the vocabulary

We create a set of unique tokens and count how large the vocabulary is before mapping each token to an integer ID.

In [12]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


## 10) Map tokens to IDs

This step creates a token-to-index dictionary so each vocabulary item receives a unique integer representation.

In [13]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab            #1
        self.int_to_str = {i:s for s,i in vocab.items()}        #2

    def encode(self, text):         #3
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):         #4
        text = " ".join([self.int_to_str[i] for i in ids]) 

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)    #5
        return text

## 11) Build a simple tokenizer class

We implement the first tokenizer version, which converts strings to token IDs and back using a fixed vocabulary.

In [18]:
tokenizer = SimpleTokenizerV1(vocab)

## 12) Instantiate the tokenizer

This creates a tokenizer object from the vocabulary built earlier so it can encode and decode text.

In [23]:
text = """"It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


## 13) Encode a sample sentence

This example converts a sentence into token IDs using the first tokenizer implementation and prints the encoded output.

In [24]:
text = """"It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable dvsdvdv pride."""

ids = tokenizer.encode(text)
print(ids)

KeyError: 'dvsdvdv'

## 14) Test a tricky example

This cell intentionally uses a phrase with apostrophes and punctuation to expose edge cases in the simple tokenizer.

In [25]:
decoded = tokenizer.decode(ids)
print(decoded)

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


## 15) Decode token IDs back to text

This demonstrates that the tokenizer can reconstruct text from integer IDs, although edge cases remain around spacing and punctuation.

In [26]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}

print(len(vocab.items()))

1132


## 16) Expand the vocabulary with special tokens

We add a closing-token marker and an unknown-token placeholder so the vocabulary is better suited for later model-style training examples.

In [27]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


## 17) Inspect the end of the vocabulary

This small check confirms that the added special tokens appear at the end of the mapping.

In [28]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int            #1
                        else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)    #2
        return text

## 18) Build the improved tokenizer

This version adds a fallback token for unknown words, making the tokenizer more robust than the first version.

In [29]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


## 19) Prepare a text example with a special token

We combine two sentences with an end-of-text marker to test how the tokenizer handles separators and special vocabulary entries.

In [30]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


## 20) Encode with the improved tokenizer

This example applies the second tokenizer version to a combined text sequence and prints the resulting IDs.

In [31]:
print(tokenizer.decode(tokenizer.encode(text)))


<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.
